In [1]:
# Thread 방식

"""
Thread 기반 동시성 예제 - 스피너(Spinner)

핵심 개념:
- threading.Thread를 이용해 스피너를 별도의 '스레드'에서 실행
- 메인 스레드는 시간이 오래 걸리는 작업(느린 계산)을 수행
- 두 스레드는 같은 프로세스 내 메모리를 공유하지만, GIL(Global Interpreter Lock)
  때문에 CPU 바운드 작업에서는 진짜 병렬 실행이 되지 않는다.
  (I/O 바운드 작업이라면 GIL이 풀리는 구간이 있어 효과적으로 동시성을 얻을 수 있음)
"""

import threading
import itertools
import time


def spin(msg: str, done: threading.Event) -> None:
    """done 이벤트가 set 될 때까지 스피너 문자를 화면에 출력한다."""
    for char in itertools.cycle(r'\|/-'):
        status = f'\r{char} {msg}'
        print(status, end='', flush=True)
        # 0.1초마다 대기하면서 done 이벤트를 체크 (즉시 반응하기 위함)
        if done.wait(0.1):
            break
    # 스피너 라인을 공백으로 지우기
    blanks = ' ' * len(status)
    print(f'\r{blanks}\r', end='')


def slow_function() -> int:
    """CPU/시간이 오래 걸리는 작업을 흉내내는 함수 (실제로는 sleep 사용)."""
    time.sleep(3)  # 실제 작업이라면 이 자리에 무거운 연산이 들어감
    return 42


def supervisor() -> int:
    done = threading.Event()
    spinner_thread = threading.Thread(target=spin, args=('작업 중...', done))
    print(f'스피너 스레드 시작: {spinner_thread}')
    spinner_thread.start()

    result = slow_function()  # 메인 스레드는 여기서 블로킹됨

    done.set()          # 스피너에게 종료 신호 전달
    spinner_thread.join()  # 스피너 스레드가 끝날 때까지 대기
    return result


def main() -> None:
    result = supervisor()
    print(f'결과: {result}')


if __name__ == '__main__':
    main()

스피너 스레드 시작: <Thread(Thread-4 (spin), initial)>
결과: 42    


In [5]:
"""
Process 기반 동시성 예제 - 스피너(Spinner)

핵심 개념:
- multiprocessing.Process를 이용해 스피너를 별도의 '프로세스'로 실행
- 각 프로세스는 독립된 메모리 공간과 자체 인터프리터(및 GIL)를 가지므로
  진짜 병렬 실행이 가능하다 (CPU 바운드 작업에 유리).
- 프로세스 간에는 메모리를 직접 공유하지 않으므로, 종료 신호 전달을 위해
  multiprocessing.Event 같은 IPC(프로세스 간 통신) 도구를 사용해야 한다.
"""

import multiprocessing
import itertools
import time


def spin(msg: str, done) -> None:
    """done 이벤트가 set 될 때까지 스피너 문자를 화면에 출력한다."""
    for char in itertools.cycle(r'\|/-'):
        status = f'\r{char} {msg}'
        print(status, end='', flush=True)
        if done.wait(0.1):
            break
    blanks = ' ' * len(status)
    print(f'\r{blanks}\r', end='')


def slow_function() -> int:
    """CPU/시간이 오래 걸리는 작업을 흉내내는 함수 (실제로는 sleep 사용)."""
    time.sleep(3)
    return 42


def supervisor() -> int:
    done = multiprocessing.Event()
    spinner_process = multiprocessing.Process(
        target=spin, args=('작업 중...', done)
    )
    print(f'스피너 프로세스 시작: {spinner_process}')
    spinner_process.start()

    result = slow_function()  # 메인 프로세스는 여기서 블로킹됨

    done.set()               # 스피너 프로세스에게 종료 신호 전달
    spinner_process.join()   # 스피너 프로세스가 끝날 때까지 대기
    return result


def main() -> None:
    result = supervisor()
    print(f'결과: {result}')


if __name__ == '__main__':
    # macOS(3.8+)는 기본 시작 방식이 'spawn'이라, 스크립트를 파일로 직접
    # 실행하지 않는 환경(Jupyter 노트북, REPL 등)에서는 자식 프로세스가
    # __main__ 모듈을 다시 import하지 못해 대상 함수를 찾지 못하는 오류가 날 수 있다.
    # 'fork' 방식은 부모 프로세스의 메모리를 그대로 복제하므로 이런 문제가 없다.
    # (fork는 macOS/Linux에서만 지원되며 Windows에서는 사용 불가)
    #
    # force=True가 필요한 이유: Jupyter 노트북에서는 커널이 계속 살아있는
    # 상태로 셀을 여러 번 실행할 수 있는데, 시작 방식은 프로세스당 한 번만
    # 설정 가능하다. 이미 설정된 상태에서 다시 호출하면
    # "RuntimeError: context has already been set"가 발생하므로,
    # force=True로 강제 재설정을 허용한다.
    multiprocessing.set_start_method('fork', force=True)
    # Windows/일부 플랫폼에서는 멀티프로세싱 시 이 가드가 필수적임
    main()

스피너 프로세스 시작: <Process name='Process-2' parent=91396 initial>
결과: 42    


In [10]:
"""
Coroutine(asyncio) 기반 동시성 예제 - 스피너(Spinner)

핵심 개념:
- 스레드나 프로세스를 새로 만들지 않고, 하나의 스레드 안에서
  '코루틴(coroutine)'들이 협력적으로(cooperatively) 실행 순서를 양보하며 동작한다.
- await 지점에서 제어권이 이벤트 루프로 넘어가고, 그 사이에 다른 코루틴이 실행된다.
- 스레드/프로세스보다 훨씬 가볍고(메모리, 컨텍스트 스위칭 비용이 적음),
  수천 개의 동시 작업도 효율적으로 처리 가능 (주로 I/O 바운드 작업에 적합).
- CPU 바운드 작업(예: 무거운 연산)에는 부적합함에 유의
  (asyncio.sleep 대신 실제 블로킹 연산을 넣으면 이벤트 루프 전체가 멈춘다).
"""

import asyncio
import itertools


async def spin(msg: str) -> None:
    """CancelledError를 받을 때까지 스피너 문자를 화면에 출력한다."""
    for char in itertools.cycle(r'\|/-'):
        status = f'\r{char} {msg}'
        print(status, end='', flush=True)
        try:
            await asyncio.sleep(0.1)
        except asyncio.CancelledError:
            break
    blanks = ' ' * len(status)
    print(f'\r{blanks}\r', end='')


async def slow_function() -> int:
    """I/O 대기를 흉내내는 코루틴 (실제로는 네트워크 호출 등이 들어갈 자리)."""
    await asyncio.sleep(3)  # 실제로는 await client.get(...) 등이 위치
    return 42


async def supervisor() -> int:
    spinner_task = asyncio.create_task(spin('작업 중...'))
    print(f'스피너 태스크 시작: {spinner_task}')

    result = await slow_function()  # 이 동안 spinner_task가 이벤트 루프에서 함께 실행됨

    spinner_task.cancel()  # 스피너 태스크에게 취소 신호 전달
    return result


def main() -> None:
    try:
        # 이미 실행 중인 이벤트 루프가 있는지 확인한다.
        # (Jupyter/IPython 노트북은 커널 자체가 이벤트 루프를 돌리고 있음)
        asyncio.get_running_loop()
    except RuntimeError:
        # 실행 중인 루프가 없음 = 일반 터미널에서 스크립트로 실행한 경우
        result = asyncio.run(supervisor())
        print(f'결과: {result}')
    else:
        # 이미 루프가 실행 중 = Jupyter 노트북 등에서 실행한 경우
        # asyncio.run()을 또 호출하면 "cannot be called from a running
        # event loop" 에러가 나므로, 노트북 셀에서는 아래처럼
        # 최상위 await를 직접 사용해야 한다:
        #
        #     result = await supervisor()
        #     print(f'결과: {result}')
        #
        raise RuntimeError(
            '이미 실행 중인 이벤트 루프가 감지되었습니다 (Jupyter 노트북 등). '
            '노트북 셀에서는 main() 대신 다음처럼 직접 실행하세요:\n'
            '    result = await supervisor()\n'
            '    print(f"결과: {result}")'
        )


# if __name__ == '__main__':
#     main()

result = await supervisor()
print(f'결과: {result}')

스피너 태스크 시작: <Task pending name='Task-62' coro=<spin() running at /var/folders/36/80byr6l16gs8rlxjr00wwr9c0000gn/T/ipykernel_91396/3699556961.py:18>>
| 작업 중...결과: 42
